#### Topic classification and paragraph splitting with LLM 

In [2]:
import os,sys
sys.path.insert(0,'../libs')
from utils import load_json
import openai
from llm_utils import BSAgent
import numpy as np
import pandas as pd
import re,json,copy
from tqdm import tqdm

##### Load API Key and run one example 

In [4]:
## load API Key
key = load_json('/root/data/keys/openai_key.json') 
os.environ['OPENAI_API_KEY'] = key['GPT_TEST']['API_KEY']
openai.api_key  = os.getenv('OPENAI_API_KEY')

# %%
llm_agent  = BSAgent(model="gpt-4o", 
                    temperature=0)
## just run one test, make sure the api works 
pt = {'System':'You are a helpful assistant.',
      'Human':'What is your name?'}
res = llm_agent.get_response_content(prompt_template=pt)
print(res)  

I don't have a personal name, but you can call me Assistant. How can I help you today?


#### Try one paragraph and see how well it works 

In [10]:
df = pd.read_csv('/root/data/Fund/CSR/topic_n_83_examples.csv')
print(df.iloc[0]['llm_topic'])
print(df.iloc[0]['examples'])
test_input = df.iloc[0]

Fiscal Consolidation
Sustaining fiscal consolidation to help achieve intergenerational equity in the medium term should guide the conduct of fiscal policy. The nonhydrocarbon balance in line with intergenerational equity remains the appropriate anchor for assessing the fiscal position in Qatar. The estimated gap between the nonhydrocarbon primary balance derived from this framework and the actual in 2018 was about 5 percentage points of non-hydrocarbon GDP. Continuation with fiscal consolidation will help to close the remaining fiscal gap in the medium term. This gradual approach is appropriate to smooth the impact of fiscal consolidation on non-oil GDP growth. Moreover, the availability of substantial fiscal space, including the sizeable accumulated assets in the sovereign wealth fund, supports a gradual fiscal consolidation and implies that—if faced with adverse shocks—the authorities can slow the consolidation or even provide temporary stimulus without endangering market access and 

#### Run one basic economic sector question and see if LLM knows basic macro

In [11]:
simple_pt = {'System':'You are an experience macroeconomist from IMF. Your job is to assign topic labels to a given paragraph from IMF document',
      'Human':'Do you kne the difference between Real sector, Fiscal sector, Monetary sector, Financial sector and External sector ?'}

res = llm_agent.get_response_content(prompt_template=simple_pt)
print(res)  

Yes, I can explain the differences between these sectors:

1. **Real Sector**:
   - **Definition**: The real sector encompasses the production and consumption of goods and services in an economy. It includes activities related to agriculture, manufacturing, services, and trade.
   - **Key Indicators**: GDP, industrial production, employment rates, and productivity.

2. **Fiscal Sector**:
   - **Definition**: The fiscal sector involves government revenue and expenditure. It includes taxation, government spending, budget deficits/surpluses, and public debt.
   - **Key Indicators**: Government budget balance, public debt-to-GDP ratio, tax revenue, and government expenditure.

3. **Monetary Sector**:
   - **Definition**: The monetary sector deals with the supply of money, interest rates, and the overall monetary policy managed by a country's central bank.
   - **Key Indicators**: Money supply (M1, M2, etc.), interest rates, inflation rates, and central bank policy rates.

4. **Financial Se

In [40]:
topic_identify_simple_pt = {
    'System':'You are an experience macroeconomist from IMF. Your job is to assign topic labels to a given paragraph from IMF document',
    'Human':"""

You are given a list of topics with their definition and key indicators as below:
----------------
----------------
1. **Real Sector**:
   - **Definition**: The real sector encompasses the production and consumption of goods and services in an economy. It includes activities related to agriculture, manufacturing, services, and trade.
   - **Key Indicators**: GDP, industrial production, employment rates, and productivity.

2. **Fiscal Sector**:
   - **Definition**: The fiscal sector involves government revenue and expenditure. It includes taxation, government spending, budget deficits/surpluses, and public debt.
   - **Key Indicators**: Government budget balance, public debt-to-GDP ratio, tax revenue, and government expenditure.

3. **Monetary Sector**:
   - **Definition**: The monetary sector deals with the supply of money, interest rates, and the overall monetary policy managed by a country's central bank.
   - **Key Indicators**: Money supply (M1, M2, etc.), interest rates, inflation rates, and central bank policy rates.

4. **Financial Sector**:
   - **Definition**: The financial sector includes institutions and markets that facilitate the flow of funds between savers and borrowers. It encompasses banks, stock markets, bond markets, and other financial intermediaries.
   - **Key Indicators**: Stock market indices, bond yields, bank lending rates, and financial stability indicators.

5. **External Sector**:
   - **Definition**: The external sector covers a country's international economic transactions, including trade in goods and services, cross-border investment, and foreign exchange markets.
   - **Key Indicators**: Trade balance, current account balance, foreign direct investment (FDI), exchange rates, and international reserves.
----------------
----------------
    
You are also given a paragraph from a report published by the International Monetary Fund as below: 
----------------
----------------
{PARA}
----------------
----------------

Please carefully analyze the paragraph and classify the provided paragraph using ONLY the provided topics. 
Try your best to assign only one topic to the paragraph. You can use multiple categories only if you are very confident that multiple topics are extensively discussed in the paragraph.
If the paragraph does not fit into any of the provided you can return 'I don't know'. 
Please provide your reasoning for your classification first, and then provide the topic label and a confidence score from 0-100.

Please respond in clean json format as follow:
reasoning: <explanation for topic label >,
topic_labels: [<topic label: confidence score>,...] or 'I don't know',

"""}

In [41]:
#print(topic_identify_simple_pt['Human'].format(PARA=test_input['examples']))

topic_pt_temp = copy.copy(topic_identify_simple_pt)
topic_pt_temp['Human']=topic_identify_simple_pt['Human'].format(PARA=test_input['examples'])
res = llm_agent.get_response_content(prompt_template=topic_pt_temp)
res_dict = llm_agent.parse_load_json_str(res) 
print(res_dict)

{'reasoning': 'The paragraph primarily discusses fiscal consolidation, fiscal policy, and fiscal indicators such as the nonhydrocarbon balance, public debt, and fiscal space. It emphasizes the importance of maintaining a sound fiscal position and prudent fiscal policies to ensure intergenerational equity and adequate savings for future generations. The focus is on government revenue and expenditure, budget deficits, and public debt, which are key indicators of the fiscal sector.', 'topic_labels': ['Fiscal Sector: 100']}


In [49]:
sample_results = []
for idx,r in tqdm(df.head(40).iterrows()):
    topic_pt_temp = copy.copy(topic_identify_simple_pt)
    topic_pt_temp['Human']=topic_identify_simple_pt['Human'].format(PARA=r['examples'])
    res = llm_agent.get_response_content(prompt_template=topic_pt_temp)
    res_dict = llm_agent.parse_load_json_str(res) 
    sample_results.append([r['llm_topic'],r['examples'],res_dict['reasoning'],res_dict['topic_labels']])

0it [00:00, ?it/s]

40it [03:49,  5.74s/it]


In [50]:
res_df = pd.DataFrame(sample_results,columns=['llm_topic','paragraph','reasoning','topic_labels'])
res_df.head()

,llm_topic,paragraph,reasoning,topic_labels
0,Fiscal Consolidation,Sustaining fiscal consolidation to help achiev...,The paragraph primarily discusses fiscal conso...,[{'Fiscal Sector': 100}]
1,Fiscal Consolidation,The authorities concurred that a return to the...,The paragraph primarily discusses fiscal polic...,[{'Fiscal Sector': 100}]
2,Fiscal Consolidation,The authorities agreed that additional fiscal ...,The paragraph primarily discusses fiscal polic...,[{'Fiscal Sector': 100}]
3,Monetary Policy,The monetary policy stance should remain broad...,The paragraph discusses the stance of monetary...,[{'Monetary Sector': 100}]
4,Monetary Policy,Greater exchange rate flexibility remains key ...,The paragraph discusses the importance of exch...,[{'External Sector': 100}]


In [51]:
res_df.to_csv('/root/data/Fund/CSR/llm_topic_results.csv')